05_ner.ipynb
Does:
Load saved model
Run classification
Run NER
Return structured output


In [12]:
%pip install spacy

Note: you may need to restart the kernel to use updated packages.


In [13]:
# # Load model
# import spacy
# nlp = spacy.load("en_core_web_md")
# # remove built-in entity_ruler (it's before ner)
# if "entity_ruler" in nlp.pipe_names:
#     nlp.remove_pipe("entity_ruler")
# # add our ruler AFTER ner so it overwrites NER output
# ruler = nlp.add_pipe("entity_ruler", after="ner", config={"overwrite_ents": True})
# ruler.add_patterns([
#     {"label": "PHONE", "pattern": [
#         {"TEXT": {"REGEX": r"^\(?\d{3}\)?\d{3}$"}},  # 410)308
#         {"TEXT": "-", "OP": "?"},                   # optional hyphen
#         {"TEXT": {"REGEX": r"^\d{4}$"}},            # 0245
#     ]},
#     {"label": "EMAIL", "pattern": [
#         {"TEXT": {"REGEX": r"^[\w\.-]+@[\w\.-]+\.\w+$"}}
#     ]},
# ])

import spacy
import joblib
import pandas as pd

# Load spaCy
nlp = spacy.load("en_core_web_md", disable=["parser", "tagger"])

# Load trained components
tfidf = joblib.load("../models/tfidf_vectorizer.pkl")
stacking_model = joblib.load("../models/stacking_model.pkl")

In [14]:
# Load your dataset
import pandas as pd
df = pd.read_csv("../data/processed/emails_with_labels.csv")
df.head()

,From,To,Subject,Date,Body,ThreadKey,Filename,year,month,clean_body,clean_subject,email_length,num_recipients,sender_domain,label
0,msagel@home.com,jarnold@enron.com,Status,2000-11-16 17:30:00+00:00,John:\n?\nI'm not really sure what happened be...,"status::Thu, 16 Nov 2000 09:30:00 -0800",36.0,2000,11,john im not really sure what happened between...,status,556,1,home.com,1
1,slafontaine@globalp.com,john.arnold@enron.com,re:summer inverses,2000-12-08 13:05:00+00:00,i suck-hope youve made more money in natgas la...,"summer inverses::Fri, 08 Dec 2000 05:05:00 -0800",19.0,2000,12,i suckhope youve made more money in natgas las...,resummer inverses,267,1,globalp.com,0
2,iceoperations@intcx.com,"icehelpdesk@intcx.com, internalmarketing@intcx...",The WTI Bullet swap contracts,2001-05-15 16:43:00+00:00,"Hi,\n\n\nFollowing the e-mail you have receive...","the wti bullet swap contracts::Tue, 15 May 200...",50.0,2001,5,hi following the email you have received yeste...,the wti bullet swap contracts,1009,2,intcx.com,1
3,jeff.youngflesh@enron.com,"anthony.gilmore@enron.com, colleen.koenig@enro...",Invitation: EBS/GSS Meeting w/Bristol Babcock ...,2000-11-27 09:49:00+00:00,Conference Room TBD.\n\nThis meeting will be t...,invitation: ebs/gss meeting w/bristol babcock ...,3.0,2000,11,conference room tbd this meeting will be to di...,invitation ebsgss meeting wbristol babcock nov...,190,4,enron.com,1
4,caroline.abramo@enron.com,mike.grigsby@enron.com,Harvard Mgmt,2000-12-12 17:33:00+00:00,Mike- I have their trader coming into the offi...,"harvard mgmt::Tue, 12 Dec 2000 09:33:00 -0800",9.0,2000,12,mike i have their trader coming into the offic...,harvard mgmt,681,1,enron.com,0


In [15]:
# text = df["Body"][0]
# doc = nlp(text)
# for ent in doc.ents:
#     print(ent.text, ent.label_)


important_df = df[df["label"] == 1].copy()


In [16]:
# def extract_entities_batch(texts):
#     results = []
    
#     for doc in nlp.pipe(texts, batch_size=50):
#         entities = {
#             "PERSON": [],
#             "ORG": [],
#             "DATE": [],
#             "TIME": [],
#             "MONEY": [],
#             "GPE": []
#         }
        
#         for ent in doc.ents:
#             if ent.label_ in entities:
#                 entities[ent.label_].append(ent.text)
        
#         results.append(entities)
    
#     return results

def extract_entities(text):
    doc = nlp(text)
    
    entities = {
        "PERSON": [],
        "ORG": [],
        "DATE": [],
        "TIME": [],
        "MONEY": [],
        "GPE": []
    }
    
    for ent in doc.ents:
        if ent.label_ in entities:
            entities[ent.label_].append(ent.text)
    
    # Remove duplicates
    return {k: list(set(v)) for k, v in entities.items()}

In [17]:
sample_df = important_df.head(1000).copy()
sample_df["entities"] = sample_df["Body"].fillna("").apply(extract_entities)


c:\Users\singh\Desktop\Sem 6\NLP\Project\email-overload-organizer\.venv\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


MemoryError: Unable to allocate 37.8 MiB for an array with shape (25828, 384) and data type float32

In [ ]:
def clean_entities(entity_dict):
    return {k: list(set(v)) for k, v in entity_dict.items()}

sample_df["entities"] = sample_df["entities"].apply(clean_entities)

In [ ]:
sample_df[["Subject", "entities"]].head()

,Subject,entities
0,Status,"{'PERSON': ['Mark Sagel', 'John'], 'ORG': [], ..."
2,The WTI Bullet swap contracts,"{'PERSON': ['Stephanie Trabia'], 'ORG': ['Admi..."
3,Invitation: EBS/GSS Meeting w/Bristol Babcock ...,"{'PERSON': [], 'ORG': ['BBI', 'EBS'], 'DATE': ..."
5,4-URGENT - OWA Please print this now.,"{'PERSON': ['Outlook', 'Printing Templates - ..."
6,Fuel Switching,"{'PERSON': ['WEFA'], 'ORG': ['FERC', 'EIA', 'D..."


In [ ]:
# def analyze_email(text):
#     # Step 1: TF-IDF
#     vec = tfidf.transform([text])
    
#     # Step 2: Classification
#     pred = stacking_model.predict(vec)[0]
    
#     if pred == 1:
#         doc = nlp(text)
        
#         entities = {}
#         for ent in doc.ents:
#             entities.setdefault(ent.label_, []).append(ent.text)
        
#         return {
#             "importance": "Important",
#             "entities": entities
#         }
    
#     else:
#         return {
#             "importance": "Not Important",
#             "entities": None
#         }




def analyze_email(text):
    
    # Step 1: Convert text → TF-IDF
    vec = tfidf.transform([text])
    
    # Step 2: Predict importance
    pred = stacking_model.predict(vec)[0]
    
    if pred == 1:
        entities = extract_entities(text)
        
        return {
            "importance": "Important",
            "entities": entities
        }
    
    else:
        return {
            "importance": "Not Important",
            "entities": None
        }

In [ ]:
email = """
Hi John, the meeting with Enron is tomorrow at 10 AM.
"""

print(analyze_email(email))

{'importance': 'Important', 'entities': {'PERSON': ['John'], 'ORG': ['Enron'], 'DATE': ['tomorrow'], 'TIME': ['10 AM'], 'MONEY': [], 'GPE': []}}


c:\Users\singh\Desktop\Sem 6\NLP\Project\email-overload-organizer\.venv\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)
